# 05c — Study cohort check of the participant flow

C06, and the running check during the participant round. A copy of
`05_pilot` with `RUN_TYPE = "study"`: run it after the **first** real
session, when an always-null field is still fixable for everyone who follows,
and again whenever the round needs a look. The checks and their code are
`05_pilot`'s (`src/pilot.py`); only the cohort differs.

Not strict, unlike the pilot: a real round need not contain a decline, a
dismissed post-task screen or a phone, so those are reported rather than
required. Early in the round, fields that need later behaviour (thumbs,
accept/reject, the post-task survey after the third turn) can still be
empty; that is a `FAIL` to read, not necessarily a fault.

Three sections, each a question the participant round cannot answer afterwards:

1. **Is every field being written?** Every field of `docs/event-model.md` that a
   participant round produces must be non-null in at least one row.
2. **Does the flow behave?** Consent, questionnaire and post-task order; the
   post-task screen fires once, after the third turn; the pinned model; labels
   and links agree with the rows they came from; the hourly `ignored` job;
   phones flagged (§4.2.2: laptop, desktop or tablet; tablets are recorded as
   `desktop`, D113).
3. **What happened?** Counts and summaries only.

The last cell raises if anything failed, listing every failure.

**Privacy.** This notebook reads participant rows. Its outputs are stripped on
commit by the `nbstripout` filter (`notebooks/study/.gitattributes`); run
`python tools/check_notebooks.py` before committing anyway. Nothing here prints
a participant's words, an id or an email: sessions appear as `S1, S2, …` in
consent order and declined ones as `D1, …`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

from datetime import datetime, timezone

import pandas as pd

from src import db, pilot

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 110)

# --- parameters -------------------------------------------------------------
RUN_TYPE = "study"            # the participant round (C06); 05_pilot covers the pilot
STRICT = RUN_TYPE == "pilot"  # pilot requires one submit, one dismiss, one decline

NOW = datetime.now(timezone.utc)
data = pilot.load(db.admin(), RUN_TYPE)
print(f"run_type={RUN_TYPE!r} · read {NOW:%Y-%m-%d %H:%M} UTC")
pd.Series({t: len(data.table(t)) for t in (
    "study_sessions", "study_survey_demographics", "study_survey_post_task", "study_suggestions",
    "study_user_actions", "study_feedback", "study_mutations")}, name="rows")

## 1. Is every field being written?

A field that exists in the schema but is always null looks correct until the
analysis needs it. **1a** checks the event model's fields under their event-model
names (where a field is stored under another name, the column is shown).
**1b** checks every column the tables actually returned, which catches
instrumentation columns the event model does not list.

`EXEMPT` rows are null by design and say why (`src/pilot.py`). In the pilot,
the post-task `submitted_at`, `dismissed_at`, `free_text` and the session's
`declined_at` are required, because C04 has one account submit, one dismiss,
and one decline.

In [ ]:
def population_frame(checks):
    return pd.DataFrame([{"table": c.table, "column": c.column, "status": c.status, "note": c.note} for c in checks])

fields = population_frame(pilot.field_population(data, strict=STRICT))
print(fields.status.value_counts().to_string())
fields[fields.status != "PASS"] if (fields.status != "PASS").any() else "1a: every event-model field is populated"

In [ ]:
columns = population_frame(pilot.column_population(data, strict=STRICT))
print(columns.status.value_counts().to_string())
columns[columns.status != "PASS"] if (columns.status != "PASS").any() else "1b: every column is populated"

## 2. Does the flow behave?

**The `ignored` job.** A suggestion nobody reacted to is stamped `ignored` by
an hourly `pg_cron` job, only once it is **24 hours** old (D17). Running the
function by hand would not help the same day, because it only touches rows
older than 24 h, and the service key cannot call it anyway: there is no
execute grant. So this check is `PENDING` until the first pilot turn is 25 h
old; it prints when to re-run. If it then fails, look at the job's history in
the Supabase SQL editor:

```sql
select status, return_message, start_time
from cron.job_run_details
where jobid = (select jobid from cron.job where jobname = 'study-finalize-ignored')
order by start_time desc limit 5;
```

`WARN` is reported and not raised: a phone session, or (outside the pilot) a
participant who left before the post-task screen appeared.

In [ ]:
flow = pd.DataFrame([vars(c) for c in pilot.flow_checks(data, STRICT, NOW)])
flow

## 3. What happened?

Pseudonymous counts only. With pilot-sized n these are a check that the
aggregates compute, not findings.

In [ ]:
pd.DataFrame(pilot.session_table(data))

In [ ]:
pd.DataFrame(pilot.turn_metrics(data)).T

In [ ]:
for name, counts in pilot.distributions(data).items():
    frame = pd.Series(counts, dtype="int64").sort_values(ascending=False)
    print(f"\n{name}\n" + (frame.to_string() if len(frame) else "  (none)"))

In [ ]:
for name, value in pilot.survey_summary(data).items():
    print(f"{name}: {dict(value) if hasattr(value, 'items') else value}")

## Verdict

Raises if any field is always null or any flow check failed, and lists all of
them. `PENDING` and `WARN` are listed and do not raise; re-run once the
`ignored` job is due.

In [ ]:
failures = [f"field {r.table}.{r.column}" for r in fields.itertuples() if r.status == "FAIL"]
failures += [f"column {r.table}.{r.column}" for r in columns.itertuples() if r.status == "FAIL"]
failures += [f"{r.name}: {r.detail}" for r in flow.itertuples() if r.status == "FAIL"]
open_items = [f"{r.status} {r.name}: {r.detail}" for r in flow.itertuples() if r.status in ("PENDING", "WARN")]

print("\n".join(open_items) or "nothing pending")
if failures:
    raise AssertionError(f"{len(failures)} failure(s):\n  " + "\n  ".join(dict.fromkeys(failures)))
print(f"PASS — {RUN_TYPE} round records every field and the flow behaves")